# LangGraph Message API Agent: Llama-3.2-1B

This notebook demonstrates a **clean LangGraph agent** built with the **Message API**, using **Llama-3.2-1B-Instruct**.

## Features

- **Message API**: Uses SystemMessage, HumanMessage, and AIMessage for proper conversation handling
- **Automatic History Management**: LangGraph's `add_messages` reducer maintains conversation history
- **No Manual Prompting**: LangChain automatically applies Llama's chat template
- **Single Model**: Simplified to use only Llama-3.2-1B-Instruct
- **Device Detection**: Automatically uses CUDA (NVIDIA GPU), MPS (Apple Silicon), or CPU
- **Graph Visualization**: Saves a visual representation of the execution graph

## How the Message API Works

Instead of manually building prompts with string concatenation, this implementation uses LangGraph's Message API:

### Traditional Approach (Bad):
```python
prompt = "User: " + user_input + "\nAssistant:"
response = llm(prompt)
```

### Message API Approach (Good):
```python
messages = [
    SystemMessage(content="You are a helpful assistant"),
    HumanMessage(content=user_input)
]
response = llm.invoke(messages)  # Returns AIMessage
```

### Benefits:

1. **Automatic History**: The `add_messages` reducer appends new messages to state
2. **Type Safety**: Message objects have well-defined structure
3. **No Post-Processing**: No need to strip "User:" or "Assistant:" markers
4. **Proper Chat Templates**: LangChain applies Llama's native chat format

## State Management

The state uses `Annotated[list, add_messages]` which:
- Automatically appends new messages instead of replacing the list
- Preserves message types (SystemMessage, HumanMessage, AIMessage)
- Maintains proper conversation order
- Requires no manual history tracking

## How to Use

1. Run all cells in order
2. Llama-3.2-1B will be loaded
3. The agent will prompt you for input
4. Type your message and press Enter
5. Llama processes your message with full conversation history
6. Type "quit", "exit", or "q" to stop the agent
7. Empty inputs will loop back to prompt without calling the LLM

## Performance Notes

- **Memory Requirements**: Llama-3.2-1B requires ~2-3GB GPU memory in fp16 mode
- **Device Support**: Works on CUDA, MPS (Apple Silicon), and CPU
- **Response Quality**: Smaller 1B model may have limited context recall

In [13]:
%pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [14]:
%pip install torch transformers langchain langchain-huggingface langgraph grandalf

In [15]:
# Import necessary libraries
import sys
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from datetime import datetime
from getpass import getpass
import operator

# Import LangChain message types for Message API
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages

In [16]:
# Determine the best available device for inference
# Priority: CUDA (NVIDIA GPU) > MPS (Apple Silicon) > CPU
def get_device():
    """
    Detect and return the best available compute device.
    Returns 'cuda' for NVIDIA GPUs, 'mps' for Apple Silicon, or 'cpu' as fallback.
    """
    if torch.cuda.is_available():
        print("Using CUDA (NVIDIA GPU) for inference")
        return "cuda"
    elif torch.backends.mps.is_available():
        print("Using MPS (Apple Silicon) for inference")
        return "mps"
    else:
        print("Using CPU for inference")
        return "cpu"

In [17]:
# =============================================================================
# STATE DEFINITION
# =============================================================================
# The state is a TypedDict that flows through all nodes in the graph.
# LangGraph's add_messages reducer automatically manages conversation history.

class AgentState(TypedDict):
    """
    State object that flows through the LangGraph nodes.
    
    Fields:
    - messages: List of conversation messages (SystemMessage, HumanMessage, AIMessage)
        Managed automatically by the add_messages reducer which:
        - Appends new messages to existing list
        - Preserves message types and order
        - Maintains full conversation history
    - should_exit: Boolean flag indicating if user wants to quit
    - skip_llm: Boolean flag to skip LLM call (for empty input)
    
    State Flow:
        START → get_user_input → call_llm → print_response → get_user_input
                     ↑__________________________________________|
                     
        Exit: get_user_input → END (when should_exit=True)
        Empty input: get_user_input → get_user_input (when skip_llm=True)
    """
    messages: Annotated[list, add_messages]
    should_exit: bool
    skip_llm: bool

In [18]:
# =============================================================================
# This cell was used for tracing utilities in the old implementation
# No longer needed with the Message API approach
# =============================================================================

In [19]:
# =============================================================================
# LLM CREATION
# =============================================================================

def create_llm():
    """
    Create and configure the Llama-3.2-1B-Instruct LLM.
    Downloads from HuggingFace Hub and wraps for LangChain usage.
    
    Returns:
        tuple: (llm, tokenizer) - LLM wrapped for LangChain and tokenizer for chat template
    """
    # Get the optimal device for inference
    device = get_device()

    # Prompt for HuggingFace token (required for gated models like Llama)
    print("\nPlease enter your HuggingFace token:")
    print("(Get your token from: https://huggingface.co/settings/tokens)")
    hf_token = getpass("HF Token: ")

    print(f"\nLoading model: meta-llama/Llama-3.2-1B-Instruct")
    print("This may take a moment on first run as the model is downloaded...")

    # Load the tokenizer - converts text to tokens the model understands
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct", token=hf_token)

    # Load the model itself with appropriate settings for the device
    model = AutoModelForCausalLM.from_pretrained(
        "meta-llama/Llama-3.2-1B-Instruct",
        token=hf_token,
        torch_dtype=torch.float16 if device != "cpu" else torch.float32,
        device_map="auto" if device == "cuda" else None,  # Use "auto" for better memory management
    )

    # Move model to MPS device if using Apple Silicon
    if device == "mps":
        model = model.to(device)

    # Create a text generation pipeline that combines model and tokenizer
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=256,  # Maximum tokens to generate in response
        do_sample=True,      # Enable sampling for varied responses
        temperature=0.7,     # Controls randomness (lower = more deterministic)
        top_p=0.95,          # Nucleus sampling parameter
        pad_token_id=tokenizer.eos_token_id,  # Suppress pad_token_id warning
    )

    # Wrap the HuggingFace pipeline for use with LangChain
    llm = HuggingFacePipeline(pipeline=pipe)

    print(f"Model loaded successfully!")
    return llm, tokenizer  # Return both LLM and tokenizer


In [ ]:
# =============================================================================
# GRAPH CREATION FUNCTION - MESSAGE API APPROACH
# =============================================================================

def create_graph(llm, tokenizer, debug=False):
    """
    Create the LangGraph state graph using Message API.
    Manually applies chat template for proper message formatting.
    
    Args:
        llm: The language model
        tokenizer: The tokenizer for chat template
        debug: Enable debug output (default: False)
    """

    # =========================================================================
    # NODES
    # =========================================================================
    def get_user_input(state: AgentState) -> dict:
        """Get user input and create HumanMessage."""
        print("\n" + "=" * 50)
        print("Enter your text (or 'quit' to exit):")
        print("=" * 50)

        print("\n> ", end="")
        sys.stdout.flush()
        user_input = input()

        if not user_input.strip():
            print("\n[Empty input - please enter some text]")
            # Set a flag to skip LLM call
            return {"messages": [], "should_exit": False, "skip_llm": True}

        print(f"\nYou: {user_input}")

        if user_input.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            return {"messages": [], "should_exit": True, "skip_llm": True}

        # Create HumanMessage - add_messages reducer will append it
        return {"messages": [HumanMessage(content=user_input)], "should_exit": False, "skip_llm": False}

    def call_llm(state: AgentState) -> dict:
        """Invoke LLM with message history, manually applying chat template."""
        messages = state["messages"]
        
        if debug:
            print(f"\n[DEBUG] State has {len(messages)} messages:")
            for i, msg in enumerate(messages):
                msg_preview = msg.content[:50] if len(msg.content) > 50 else msg.content
                print(f"  {i}: {type(msg).__name__} - {msg_preview}...")

        # Defensive check: add system message if somehow missing from state
        # (System message should already be in initial state, so this is a safeguard)
        if not any(isinstance(m, SystemMessage) for m in messages):
            messages = [SystemMessage(content="You are a helpful AI assistant. Respond to the user's message with a single, direct response.")] + messages

        # Convert LangChain messages to the format apply_chat_template expects
        formatted_messages = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                formatted_messages.append({"role": "system", "content": msg.content})
            elif isinstance(msg, HumanMessage):
                formatted_messages.append({"role": "user", "content": msg.content})
            elif isinstance(msg, AIMessage):
                formatted_messages.append({"role": "assistant", "content": msg.content})
        
        if debug:
            print(f"\n[DEBUG] Formatted {len(formatted_messages)} messages for chat template")

        # Apply Llama-3.2's chat template to format the prompt correctly
        prompt = tokenizer.apply_chat_template(
            formatted_messages,
            tokenize=False,
            add_generation_prompt=True  # Adds the assistant prompt at the end
        )
        
        if debug:
            print(f"\n[DEBUG] Prompt length: {len(prompt)} chars")
            print(f"[DEBUG] Prompt preview:\n{prompt[:300]}...")

        # Invoke LLM with the formatted prompt string
        print("\n[Processing...]")
        sys.stdout.flush()  # Flush before LLM call
        
        full_response = llm.invoke(prompt)
        
        if debug:
            print(f"\n[DEBUG] Full response length: {len(full_response)} chars")
            print(f"[DEBUG] Full response preview: {full_response[:100]}...")
        
        # Extract only the new completion (remove the prompt)
        # The response includes the full prompt + completion
        if len(full_response) < len(prompt):
            if debug:
                print(f"\n[ERROR] Response is shorter than prompt! Response: {len(full_response)}, Prompt: {len(prompt)}")
            completion = full_response.strip()
        else:
            completion = full_response[len(prompt):].strip()
        
        if debug:
            print(f"\n[DEBUG] Extracted completion ({len(completion)} chars): {completion[:100]}...")
            sys.stdout.flush()  # Flush debug output

        # Wrap in AIMessage
        return {"messages": [AIMessage(content=completion)], "skip_llm": False}

    def print_response(state: AgentState) -> dict:
        """Print the most recent AI response."""
        messages = state["messages"]
        
        if debug:
            print(f"\n[DEBUG print_response] Total messages in state: {len(messages)}")
            sys.stdout.flush()
        
        if len(messages) == 0:
            print("[ERROR] No messages in state!")
            return {"skip_llm": False}
        
        last_message = messages[-1]
        
        if debug:
            print(f"[DEBUG print_response] Last message type: {type(last_message).__name__}")
            print(f"[DEBUG print_response] Last message content length: {len(last_message.content)}")
            print(f"[DEBUG print_response] Last message content preview: {last_message.content[:50]}...")
            sys.stdout.flush()

        if isinstance(last_message, AIMessage):
            print("\n" + "=" * 80)
            print("[LLAMA-3.2-1B-INSTRUCT]")
            print("-" * 80)
            print(last_message.content)
            print("=" * 80)

            sys.stdout.flush()
            time.sleep(0.1)
            print()
            sys.stdout.flush()
        else:
            print(f"[WARNING] Last message is not AIMessage, it's {type(last_message).__name__}")

        return {"skip_llm": False}  # Reset the flag

    # =========================================================================
    # CONDITIONAL ROUTING
    # =========================================================================
    def route_after_input(state: AgentState) -> str:
        """Route based on whether user wants to exit or skip LLM."""
        if state.get("should_exit", False):
            return END
        elif state.get("skip_llm", False):
            # Skip LLM and go directly back to input
            return "get_user_input"
        else:
            return "call_llm"

    # =========================================================================
    # GRAPH CONSTRUCTION
    # =========================================================================
    graph_builder = StateGraph(AgentState)

    # Add nodes
    graph_builder.add_node("get_user_input", get_user_input)
    graph_builder.add_node("call_llm", call_llm)
    graph_builder.add_node("print_response", print_response)

    # Add edges
    graph_builder.add_edge(START, "get_user_input")
    
    # Conditional routing after user input
    graph_builder.add_conditional_edges(
        "get_user_input",
        route_after_input,
        {
            "call_llm": "call_llm",
            "get_user_input": "get_user_input",  # Loop back for empty input
            END: END
        }
    )
    
    # Linear flow: call_llm → print_response → loop back to get_user_input
    graph_builder.add_edge("call_llm", "print_response")
    graph_builder.add_edge("print_response", "get_user_input")

    return graph_builder.compile()

In [21]:
# =============================================================================
# GRAPH VISUALIZATION
# =============================================================================

def save_graph_image(graph, filename="lg_graph.png"):
    """
    Generate a Mermaid diagram of the graph and save it as a PNG image.
    Uses the graph's built-in Mermaid export functionality.
    """
    try:
        # Get the Mermaid PNG representation of the graph
        # This requires the 'grandalf' package for rendering
        png_data = graph.get_graph(xray=True).draw_mermaid_png()
        
        # Write the PNG data to file
        with open(filename, "wb") as f:
            f.write(png_data)
        
        print(f"Graph image saved to {filename}")
    except Exception as e:
        print(f"Could not save graph image: {e}")
        print("You may need to install additional dependencies: pip install grandalf")

In [ ]:
# =============================================================================
# MAIN FUNCTION - MESSAGE API APPROACH
# =============================================================================

def main(debug=False):
    """
    Main function that orchestrates the single-model agent using Message API.
    
    Args:
        debug: Enable debug output (default: False)
    """
    print("=" * 80)
    print("LangGraph Message API Agent: Llama-3.2-1B")
    print("=" * 80)
    print()

    # Load the LLM and tokenizer
    llm, tokenizer = create_llm()

    print("\nCreating LangGraph with Message API...")
    graph = create_graph(llm, tokenizer, debug=debug)
    print("Graph created successfully!")

    print("\nSaving graph visualization...")
    save_graph_image(graph)

    # Initialize state with system message and empty conversation history
    initial_state: AgentState = {
        "messages": [SystemMessage(content="You are a helpful AI assistant. Respond to the user's message with a single, direct response.")],
        "should_exit": False,
        "skip_llm": False
    }

    print(f"\nStarting agent with Message API...")
    print("The conversation history is managed automatically using add_messages reducer.")
    print("Type 'quit', 'exit', or 'q' to stop.\n")

    graph.invoke(initial_state)

In [ ]:
# =============================================================================
# RUN THE AGENT
# =============================================================================
# To enable debug output, call main(debug=True) instead

if __name__ == "__main__":
    main()  # Set debug=True for detailed debugging output